# Fermionic Fractional Chern Insulator on the Checkerboard Lattice

**Abstract.** This notebook is a pedagogical tour of the **fermionic fractional Chern insulator**
(FCI) on the checkerboard lattice — the Sun–Gu–Katsura–Das Sarma (SGKS) model, whose two flat bands
carry Chern numbers $\pm 1$ and whose interacting ground state at fractional band filling is a
fermionic Laughlin-type FCI. We derive the staggered-flux tight-binding model, discuss the two flat
Chern bands, and then run symmetry-resolved ED for spinless fermions. At the fine-tuned
$\nu=1/3$ band filling (4 fermions on the $3\times4$ torus, $t''=-0.2$, interaction scale
$\lambda=2.8$) we reproduce the Julia-reference energies to $\lesssim 2\times10^{-13}$ and identify
a **three-fold near-degenerate ground state** (GSD$=3$). We then demonstrate the **spectrum flow**
and the **fractional charge pump** ($\Delta Q\approx 2/3$) at the particle-hole-related $\nu=2/3$
filling (8 fermions), which is the package's canonical fermionic pump.

**References.**
1. K. Sun, Z. Gu, H. Katsura, S. Das Sarma, *Nearly flatbands with nontrivial topology*, Phys. Rev. Lett. **106**, 236803 (2011); arXiv:1012.5864.
2. E. Tang, J.-W. Mei, X.-G. Wen, *High-temperature fractional quantum Hall states*, Phys. Rev. Lett. **106**, 236802 (2011).
3. T. Neupert, L. Santos, C. Chamon, C. Mudry, *Fractional quantum Hall states at zero magnetic field*, Phys. Rev. Lett. **106**, 236804 (2011).
4. N. Regnault, B. A. Bernevig, *Fractional Chern insulator*, Phys. Rev. X **1**, 021014 (2011).


## The Checkerboard Lattice with Staggered Flux

The checkerboard lattice is a square lattice decorated with two sublattices per unit cell:

\begin{equation}
\mathbf a_1=(1,0),\quad \mathbf a_2=(0,1),\qquad
\boldsymbol\delta_1=(1/2,0),\quad \boldsymbol\delta_2=(0,1/2).
\end{equation}

Each square plaquette is threaded by a **staggered** flux: nearest-neighbor (inter-sublattice) hops
carry phases $\pm\phi$ arranged so that the flux through adjacent plaquettes alternates in sign, with
zero net flux per unit cell. Time-reversal symmetry is broken, yet the model is a purely
lattice-hopping Hamiltonian with no external magnetic field.

### Hopping terms (NN / NNN / NNNN)

Following SGKS, the tight-binding model has three hopping ranges (all with a global sign chosen to
make the lower band flat):

- **NN** (inter-sublattice, complex): $-t\,e^{\pm i\phi}$ with $\phi=2\pi/8=\pi/4$ (the staggered
  flux).
- **NNN** (intra-sublattice, real, anisotropic): $-t'_1$ along $x$ and $-t'_2$ along $y$, with
  $t'_1=-t'_2=1/(2+\sqrt2)$.
- **NNNN** (intra-sublattice, diagonal, real): $-t''$ with $t''=1/(2+2\sqrt2)$.

The fine-tuned NNN/NNNN amplitudes make both bands **exactly flat** in the SGKS parameter window,
while the staggered flux endows them with Chern numbers $C=\pm 1$.


## Two Flat Chern Bands

In momentum space the two-band Hamiltonian is $H(\mathbf k)=\mathbf d(\mathbf k)\cdot\boldsymbol\sigma$
(the Pauli matrices acting on the sublattice degree of freedom). The flatness means the single-particle
gap $2|\mathbf d(\mathbf k)|$ is $\mathbf k$-independent, so the bandwidth vanishes and interactions
dominate entirely — the ideal setting for an FCI. Filling the lower flat band ($C=+1$) to a rational
fraction $\nu$ and adding interactions can realize a fractional quantum Hall state with no Landau
levels.

> **Band-filling vs. vertex-filling.** The ED engine works on the flattened graph of $2\,L_x L_y$
> vertices. "$\nu=1/3$ band filling" means 1/3 of the lower band's $L_xL_y$ states are occupied, i.e.
> $N_e = L_xL_y/3 = 4$ fermions on $[3,4]$ — a **vertex filling** of $4/24=1/6$. Likewise
> "$\nu=2/3$ band filling" is $N_e = 2L_xL_y/3 = 8$ fermions (vertex filling $1/3$). The two are
> related by particle–hole symmetry of the two-band system.


## The Interacting Model

For spinless fermions we add density–density interactions between the first three neighbor shells:

\begin{equation}
H = H_0 + V_1\!\!\sum_{\langle i,j\rangle} n_i n_j + V_2\!\!\sum_{\langle\!\langle i,j\rangle\!\rangle} n_i n_j
       + V_3\!\!\sum_{\langle\!\langle\!\langle i,j\rangle\!\rangle\!\rangle} n_i n_j ,
\qquad n_i = c_i^\dagger c_i .
\end{equation}

The SGKS parameters are $(V_1,V_2,V_3)=(2,1,0)$. For the fine-tuned FCI we also rescale all
interactions by $\lambda=2.8$ and set $t''=-0.2$ (the `my_optimal_param` phase point), which
stabilizes a robust $\nu=1/3$ FCI with a clear many-body gap.


## ED at $\nu=1/3$ (4 fermions, $t''=-0.2$): the GSD = 3 Multiplet

We build the fine-tuned model through `build_phase_explore_fermionic_checkerboard_model` (which
returns `(model, ed_data, n_filled, filling)`), resolve the $\mathbb Z_3\times\mathbb Z_4$
translation symmetry, and scan all 12 momentum sectors.


In [ ]:
import math
import os
from fractions import Fraction
import numpy as np
import realspace_exactdiagonalization_py as ed

PROJECT_ROOT = os.path.dirname(os.path.dirname(os.path.abspath(ed.__file__)))
FIG_DIR = os.path.join(PROJECT_ROOT, "doc", "figures")
os.makedirs(FIG_DIR, exist_ok=True)

def sector_vals(ed_data, k):
    for idx, irrep in enumerate(ed_data.irrep_list):
        if irrep.label == k:
            return ed_data.ed_scan_res[idx][0]   # 0-based key
    raise KeyError(k)

# Fine-tuned ν=1/3 FCI parameters: my_optimal_param with t'' = -0.2
params = dict(ed.models.fermionic_fci.my_optimal_param)
params["t''"] = -0.2

model, ed_data, n_filled, filling = ed.build_phase_explore_fermionic_checkerboard_model(
    params, [3, 4], Fraction(1, 3))
print(f"n_filled = {n_filled} fermions on {model.lattice.n_site} vertices "
      f"(vertex filling = {filling})")
print(f"Full Hilbert space dim: {math.comb(model.lattice.n_site, n_filled)}")
print(f"Orbits: {len(ed_data.orbit_catalog.representative_mask_list)}")
print(f"Irreps: {len(ed_data.irrep_list)}  (momenta (k1,k2))")

ed.ed_scan(ed_data, nev=5, mode="matrix")

for k in [(0, 2), (0, 0), (1, 0)]:
    vals = [float(v) for v in sector_vals(ed_data, k)[:5]]
    print(f"k={k}: {[f'{v:.12f}' for v in vals]}")

# lowest eigenvalues across all sectors
all_vals = sorted(
    (float(v), ed_data.irrep_list[idx].label)
    for idx in ed_data.ed_scan_res for v in ed_data.ed_scan_res[idx][0])
print("\nLowest 5 (energy, sector):")
for e, lbl in all_vals[:5]:
    print(f"  E = {e:.10f}  (k = {lbl!r})")

fig, ax = ed.plot_spectrum(ed_data, shift_to_zero=True)
fig.savefig(os.path.join(FIG_DIR, "fermionic_FCI_spectrum_34_nu13.svg"))
print("saved spectrum → doc/figures/fermionic_FCI_spectrum_34_nu13.svg")


**Verified alignment numbers** (Julia ARPACK, `nev=5`; Python agrees to $<2\times10^{-13}$):

| Sector | lowest five eigenvalues |
|---|---|
| $k=(0,2)$ | $-6.832285968463585$, $-6.488761638231560$, $-5.835575203799947$, $-5.713457341901553$, $-5.572292581119391$ |
| $k=(0,0)$ | $-6.536696664020559$, $-6.377168957543855$, $-6.339387043488890$, $-6.192010626043237$, $-5.495381320129844$ |
| $k=(1,0)$ | $-6.533812748546839$, $-6.512852966567995$, $-6.266923917001573$, $-6.084898163737772$, $-5.557311700956236$ |

The global ground state is at $k=(0,2)$; together with the degenerate states at $k=(1,2)$ and
$k=(2,2)$ (the $k_y=2$ momentum row) it forms the three-fold near-degenerate FCI multiplet
(GSD$=3$), split by finite-size effects.


## Spectrum Flow and Charge Pump at $\nu=2/3$ ($\Delta Q\approx 2/3$)

The package's canonical fermionic charge pump is the **$\nu=2/3$** FCI (8 fermions on $[3,4]$, the
standard SGKS parameters): its three nearly-degenerate ground states live at $k=(0,0),(1,0),(2,0)$,
and each polarization branch winds by $\Delta Q\approx 2/3$ per flux quantum (total $2$). By
particle–hole symmetry this is the partner of the $\nu=1/3$ state above (whose pump winds by
$1/3$ per branch, total $1$). We scan 7 flux points with `nev=3` (default 9) to keep the cells
snappy.


In [ ]:
import os
from fractions import Fraction
import numpy as np
import realspace_exactdiagonalization_py as ed

PROJECT_ROOT = os.path.dirname(os.path.dirname(os.path.abspath(ed.__file__)))
FIG_DIR = os.path.join(PROJECT_ROOT, "doc", "figures")
CKPT_DIR = os.path.join(PROJECT_ROOT, "doc", "checkpoints")
os.makedirs(FIG_DIR, exist_ok=True)

sample_size = [3, 4]
model = ed.build_zero_flux_fermionic_fci_second_quantized_model(
    sample_size=sample_size, params=ed.params_Sun_Gu_Katsura_Sarma)
labels = ed.default_fci_sectors_fermionic(sample_size)   # [(0,0),(1,0),(2,0)]
flux_list = list(np.linspace(0.0, 1.0, 7))

flow = ed.flux_spectrum_flow(
    model, labels,
    filling_fraction=Fraction(1, 3),      # 8 fermions / 24 vertices (ν=2/3 per band)
    flux_direction=1,
    twisted_phases_over_2π_list=flux_list,
    nev=3,
    fig_path=os.path.join(FIG_DIR, "fermionic_FCI_spectrum_flow_34_nu23.svg"),
    checkpoint_dir=CKPT_DIR,
)
print("ground-state energy per flux point, per sector:")
print(flow.energies[:, :, 0])


In [ ]:
import os
from fractions import Fraction
import numpy as np
import realspace_exactdiagonalization_py as ed

PROJECT_ROOT = os.path.dirname(os.path.dirname(os.path.abspath(ed.__file__)))
FIG_DIR = os.path.join(PROJECT_ROOT, "doc", "figures")
CKPT_DIR = os.path.join(PROJECT_ROOT, "doc", "checkpoints")
os.makedirs(FIG_DIR, exist_ok=True)

sample_size = [3, 4]
model = ed.build_zero_flux_fermionic_fci_second_quantized_model(
    sample_size=sample_size, params=ed.params_Sun_Gu_Katsura_Sarma)
labels = ed.default_fci_sectors_fermionic(sample_size)   # [(0,0),(1,0),(2,0)]
flux_list = list(np.linspace(0.0, 1.0, 7))

pump = ed.flux_charge_pump(
    model, labels,
    filling_fraction=Fraction(1, 3),
    flux_direction=1,              # thread θ along x
    polarization_direction=2,      # measure U_y (transverse)
    twisted_phases_over_2π_list=flux_list,
    nev_per_sector=1,
    fig_path=os.path.join(FIG_DIR, "fermionic_FCI_charge_pump_34_nu23.svg"),
    checkpoint_dir=CKPT_DIR,
)
print("pumped charges ΔQ =", pump.pumped_charges)
print("|ΔQ| per branch ≈ 2/3 → total =", round(abs(pump.pumped_charges).sum(), 6))


The three branches each pump $\Delta Q\approx 2/3$ of a charge per flux quantum, summing to
$2$ — the integer many-body Chern number of the $\nu=2/3$ FCI. This is the finite-size fingerprint
of the $\nu=2/3$ fermionic Laughlin state (GSD$=3$ on the torus).

---

*This notebook is part of `realspace_exactdiagonalization_py`. The model builders live in
`models/fermionic_fci.py`; the observables in `observables/spectrum_flow.py` and
`observables/charge_pump.py`.*
